In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader

# Detectar Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Executando no Google Colab")
except:
    IN_COLAB = False
    print("✓ Executando localmente")

# Setup Path
if IN_COLAB:
    if not os.path.exists('/content/ufc-easytpp'):
        print("Clonando repositório...")
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
else:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- CORREÇÃO CRÍTICA PARA AMP (FP16) ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
import math
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        # CORREÇÃO: -1e4 em vez de -1e9 para evitar NaN em FP16
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

# 1. Patch no módulo original
baselayer.attention = attention_fixed

# 2. Patch nos módulos que já importaram a função
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except ImportError: pass
try:
    import easy_tpp.model.torch_model.torch_rothp_hybrid
    easy_tpp.model.torch_model.torch_rothp_hybrid.attention = attention_fixed
except ImportError: pass

print("✓ Patch FP16 aplicado com sucesso")
# --------------------------------------

from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_thp import THP
from easy_tpp.model.torch_model.torch_rothp_hybrid import RoTHPHybrid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list, pad_id, time_scale):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        
        ts = (ts - ts[0]) / time_scale
        td = td / time_scale
        
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)


In [ ]:
def train_model(model_cls, model_name, dataset_name, train_data, dev_data, num_types, pad_id, time_scale, epochs=50):
    config = ModelConfig(num_types, pad_id)
    model = model_cls(config).to(device)
    checkpoint_path = f'best_{model_name}_{dataset_name}.pth'
    
    # Resume logic
    if os.path.exists(checkpoint_path):
        print(f"\n>>> Carregando {model_name} pré-treinado para {dataset_name}...")
        model.load_state_dict(torch.load(checkpoint_path))
        return model
        
    print(f"\n>>> Treinando {model_name} em {dataset_name} | Epochs: {epochs}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
    train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate, num_workers=0)
    dev_loader = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate, num_workers=0)
    
    best_val_nll = float('inf')
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        total_events = 0
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num_events = model.loglike_loss(batch)
                loss_norm = loss / (num_events + 1e-9)
            scaler.scale(loss_norm).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            total_events += num_events
            
        model.eval()
        val_loss = 0
        val_events = 0
        with torch.no_grad():
            for batch in dev_loader:
                batch = [t.to(device) for t in batch]
                with torch.amp.autocast('cuda'):
                    loss, num_events = model.loglike_loss(batch)
                val_loss += loss.item()
                val_events += num_events
        
        val_nll = val_loss / (val_events + 1e-9)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Ep {epoch}: Val NLL {val_nll:.4f}")
        
        if val_nll < best_val_nll:
            best_val_nll = val_nll
            torch.save(model.state_dict(), checkpoint_path)
            
    model.load_state_dict(torch.load(checkpoint_path))
    return model


In [ ]:
def evaluate_with_shift(model, test_data, shifts, pad_id, time_scale):
    collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
    loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate, num_workers=0)
    
    results = []
    model.eval()
    
    for shift in shifts:
        total_nll = 0
        total_acc = 0
        total_rmse = 0
        total_events = 0
        
        with torch.no_grad():
            for batch in loader:
                batch_gpu = [t.to(device) for t in batch]
                
                # --- APLICAR SHIFT --- 
                shift_norm = shift / time_scale
                batch_gpu[0] = batch_gpu[0] + shift_norm
                
                _, time_delta_target, type_target, mask_target, _ = batch_gpu
                
                with torch.amp.autocast('cuda'):
                    loss, num_events = model.loglike_loss(tuple(batch_gpu))
                total_nll += loss.item()
                
                dtimes_pred, types_pred = model.predict_one_step_at_every_event(tuple(batch_gpu))
                
                target_types = type_target[:, 1:]
                target_deltas = time_delta_target[:, 1:]
                target_mask = mask_target[:, 1:]
                
                correct = (types_pred == target_types) * target_mask
                total_acc += correct.sum().item()
                se = ((dtimes_pred - target_deltas) ** 2) * target_mask
                total_rmse += se.sum().item()
                total_events += target_mask.sum().item()
        
        metrics = {
            'shift': shift,
            'nll': total_nll / (total_events + 1e-9),
            'acc': total_acc / (total_events + 1e-9),
            'rmse': np.sqrt(total_rmse / (total_events + 1e-9))
        }
        results.append(metrics)
        
    return pd.DataFrame(results)


In [ ]:
datasets_to_test = ['amazon', 'taobao', 'stackoverflow']
shifts_to_test = [0, 10, 100, 1000, 10000]
models_to_test = [
    ('RoTHP', RoTHP),
    ('RoTHP Hybrid', RoTHPHybrid),
    ('THP', THP)
]

all_results = {} # {dataset: {model: df}}

for ds_name in datasets_to_test:
    all_results[ds_name] = {}
    try:
        print(f"\n{'='*60}\nDATASET: {ds_name}\n{'='*60}")
        
        try:
            dataset = load_dataset(f"easytpp/{ds_name}")
        except Exception as e:
            print(f"! Falha ao carregar {ds_name}. Usando 'retweet'...")
            dataset = load_dataset("easytpp/retweet")

        train_data = dataset['train']
        dev_data = dataset['validation']
        test_data = dataset['test']
        
        all_deltas = []
        for item in train_data:
            td = item['time_since_last_event']
            all_deltas.extend([d for d in td if d > 0])
        time_scale = np.mean(all_deltas)
        
        max_type = 0
        for x in train_data:
            max_type = max(max_type, max(x['type_event']))
        num_types = max_type + 1
        pad_id = num_types
        
        # Loop por Modelo
        for model_name, model_cls in models_to_test:
            # Treino (Normal)
            model = train_model(model_cls, model_name, ds_name, train_data, dev_data, num_types, pad_id, time_scale, epochs=50)
            
            # Avaliação (Shift)
            print(f"  Avaliar shifts para {model_name}...")
            df = evaluate_with_shift(model, test_data, shifts_to_test, pad_id, time_scale)
            all_results[ds_name][model_name] = df
            print(df.head())
        
    except Exception as e:
        print(f"ERRO CRÍTICO {ds_name}: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*40)
print("GRID SEARCH FINALIZADO")
print("="*40)


In [ ]:
# Plotar Comparação
fig, axes = plt.subplots(len(all_results), 3, figsize=(24, 5 * len(all_results)))
if len(all_results) == 1: axes = axes.reshape(1, -1)

metrics_map = {'nll': 'NLL', 'acc': 'Accuracy', 'rmse': 'RMSE'}

for i, (ds_name, models_res) in enumerate(all_results.items()):
    if not models_res: continue
    
    # Plot NLL, Acc, RMSE
    for j, (metric, label) in enumerate(metrics_map.items()):
        ax = axes[i, j]
        for model_name, df in models_res.items():
            ax.plot(df['shift'], df[metric], marker='o', label=model_name, linewidth=2)
        
        ax.set_title(f'{ds_name}: {label} vs Shift')
        ax.set_xscale('symlog')
        ax.set_xlabel('Time Shift')
        ax.set_ylabel(label)
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
